In [1]:
!pip install -q langchain-google-genai langchain-community langchain chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 116.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204

In [2]:
# IMPORT LIBRARIES

import os
import getpass as getpass
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings , ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

/tmp/ipykernel_508/3844868573.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [3]:
def setup_env():
  # check if API Key already exists
  if not os.getenv("GOOGLE_API_KEY"):
    # Ask the user to enter the API key
    os.environ['GOOGLE_API_KEY'] = getpass.getpass('Enter your API key -')

setup_env()

Enter your API key -··········


In [4]:
import sys

In [5]:
# CREATE A FUNCTION FOR LOADING THE DOCUMENT AND THEN SPLITING (CHUNKING)

def load_and_split(filepath):
  if not os.path.exists(filepath): # Check whether the file actually exists or not
    print(f'Error: File not found at {filepath}')
    sys.exit(1)

  print(f'LOADING THE DATA')
  loader = TextLoader(filepath)
  docs = loader.load()

  print(f'SPLITTING THE DATA INTO CHUNKS')
  splitter = RecursiveCharacterTextSplitter(chunk_size = 500 , chunk_overlap=200)
  splits = splitter.split_documents(docs)
  print(f'Split {len(splits)} chunks')
  return splits

In [6]:
# CREATE A RAG CHAIN INSIDE THIS WE WILL FIRST PERFORM EMBEDDING AND VECTOR STORE

def create_rag_chain(splits):
  print(f'Embedding -> Initialize vector store -> create rag chain')
  embeddings = GoogleGenerativeAIEmbeddings(model = "gemini-embedding-001",task_type = "retrieval_document")
  # Now we are going to pass embedding to our vector store/database
  vectorstore = Chroma.from_documents(documents=splits , embedding=embeddings)
  retriever = vectorstore.as_retriever()

  # Since our retriever is ready , we will create our LLM
  llm = ChatGoogleGenerativeAI(model = "gemini-3.5-flash" , temperature=0 )

  template = """Answer the question based only on the following context : {context}
  Question : {question}

  Helpful Answer :"""
  prompt = ChatPromptTemplate.from_template(template)

  # We are creating doc for the content pages

  def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

  chain = (
      {"context": retriever | format_docs , 'question':RunnablePassthrough()}
      | prompt
      | llm
      | StrOutputParser()
  )
  return chain


In [7]:
setup_env()
filepath = '/content/harrypotter.txt'

In [8]:
splits = load_and_split(filepath)
rag_chain = create_rag_chain(splits)

LOADING THE DATA
SPLITTING THE DATA INTO CHUNKS
Split 15 chunks
Embedding -> Initialize vector store -> create rag chain


In [9]:
# Gradio
!pip install gradio

In [10]:
# Gradio setup
import gradio as gr
def respond(message,chat_history):
  try:
    return rag_chain.invoke(message)
  except Exception as e:
    return f'An error occured {e}'


demo = gr.ChatInterface(
    fn=respond,
    textbox = gr.Textbox(placeholder='Ask a question related to Harry Potter',container=False,scale=7)
)
demo.launch(share=True,debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://0a815e6d117e1071f3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://0a815e6d117e1071f3.gradio.live
